In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

train_df = pd.read_csv('train_preprocessed.csv')
test_df = pd.read_csv('test_preprocessed.csv')

X_train = train_df.drop('RainTomorrow', axis=1)
y_train = train_df['RainTomorrow']
X_test = test_df.drop('RainTomorrow', axis=1)
y_test = test_df['RainTomorrow']



def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
 
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec  = recall_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)
    cm   = confusion_matrix(y_te, y_pred)
 
    return {
        'name':      name,
        'model':     model,
        'y_pred':    y_pred,
        'accuracy':  acc,
        'precision': prec,
        'recall':    rec,
        'f1':        f1,
        'cm':        cm,
    }

In [2]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
res_lr = evaluate('Logistic Regression', lr,
                   X_train, y_train, X_test, y_test)

In [3]:
# Linear Discriminant Analysis
lda = LinearDiscriminantAnalysis()
res_lda = evaluate('Linear Discriminant Analysis', lda,
                    X_train, y_train, X_test, y_test)

In [4]:
results = [res_lr, res_lda]

results_df = pd.DataFrame([{
    'Model': r['name'],
    'Accuracy': r['accuracy'],
    'Precision': r['precision'],
    'Recall': r['recall'],
    'F1': r['f1']
} for r in results])

print(results_df)

                          Model  Accuracy  Precision    Recall        F1
0           Logistic Regression  0.794613   0.528636  0.773176  0.627938
1  Linear Discriminant Analysis  0.797532   0.533573  0.769098  0.630044


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, res in zip(axes, results):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Rain', 'Rain'],
                yticklabels=['No Rain', 'Rain'])
    ax.set_title(res['name'])
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices_lr_lda.png')
plt.close()